# 默认 MinIO 测试数据集构建

将 `C:\\Users\\wuchaoli\\Pictures\\5月22日小车采集图（原始数据）` 中的图片导入 MinIO，并生成 full 与 sample_1000 两个 raw Dataset。

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path
from urllib.parse import urlparse

from dotenv import load_dotenv

from image_gallery.dataset import Dataset
from image_gallery.importers import ImportPipeline
from image_gallery.importers.local_path import SUPPORTED_IMAGE_SUFFIXES
from image_gallery.storage import MinioStorage

In [2]:
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if not (repo_root / "pyproject.toml").exists():
    raise RuntimeError("cannot locate repository root from current working directory")

load_dotenv(repo_root / ".env")

source_dir = Path("/mnt/c/Users/wuchaoli/Pictures/5月22日小车采集图（原始数据）")
if not source_dir.exists():
    raise RuntimeError(f"source directory does not exist: {source_dir}")

library_root = repo_root / "notebooks" / ".importers_test_library" / "default_minio_dataset"
full_dir = library_root / "full"
sample_dir = library_root / "sample_1000"
sample_size = 1000
random_state = 20260706

image_paths = sorted(
    path
    for path in source_dir.rglob("*")
    if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_SUFFIXES
)
if len(image_paths) < sample_size:
    raise RuntimeError(f"source image count {len(image_paths)} is less than sample size {sample_size}")

print("repo_root:", repo_root)
print("source_dir:", source_dir)
print("source_image_count:", len(image_paths))

repo_root: /home/wuchaoli/codespace/ImageGallery
source_dir: /mnt/c/Users/wuchaoli/Pictures/5月22日小车采集图（原始数据）
source_image_count: 16565


In [3]:
def require_env(name: str) -> str:
    """读取必需环境变量，缺失时立即失败。"""
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(f"missing required environment variable: {name}")
    return value


def parse_minio_endpoint(raw_endpoint: str) -> tuple[str, bool]:
    """把 .env 中的 endpoint 拆分为 MinIO SDK 需要的 host 和 secure 标记。"""
    parsed = urlparse(raw_endpoint)
    if parsed.scheme in {"http", "https"}:
        return parsed.netloc, parsed.scheme == "https"
    return raw_endpoint, False


raw_endpoint = require_env("IMAGE_GALLERY_MINIO_ENDPOINT")
access_key = require_env("IMAGE_GALLERY_MINIO_ACCESS_KEY")
secret_key = require_env("IMAGE_GALLERY_MINIO_SECRET_KEY")
bucket = require_env("IMAGE_GALLERY_MINIO_BUCKET")
endpoint, secure = parse_minio_endpoint(raw_endpoint)

storage = MinioStorage(storage_name="default_minio_test_dataset").connect(
    endpoint=endpoint,
    access_key=access_key,
    secret_key=secret_key,
    bucket=bucket,
    secure=secure,
)

print("bucket:", bucket)
print("secure:", secure)

bucket: test
secure: False


In [4]:
result = ImportPipeline(
    source=source_dir,
    storage=storage,
    output_dir=full_dir,
    global_tags=[
        "dataset/default_minio",
        "dataset/full",
        "source/windows_pictures",
        "source/vehicle_20240522",
        "storage/minio",
    ],
).run()

print("full_raw_dataset_path:", result.raw_dataset_path)
print("full_import_report_path:", result.import_report_path)
print("full_failure_manifest_path:", result.failure_manifest_path)
print("full_import_report:", result.report)

full_raw_dataset_path: /home/wuchaoli/codespace/ImageGallery/notebooks/.importers_test_library/default_minio_dataset/full/raw.parquet
full_import_report_path: /home/wuchaoli/codespace/ImageGallery/notebooks/.importers_test_library/default_minio_dataset/full/import_report.json
full_failure_manifest_path: /home/wuchaoli/codespace/ImageGallery/notebooks/.importers_test_library/default_minio_dataset/full/failure_manifest.jsonl
full_import_report: {'success_count': 16160, 'failure_count': 405}


In [5]:
full_dataset = Dataset.load(result.raw_dataset_path)
full_frame = full_dataset.to_frame()
if len(full_frame) < sample_size:
    raise RuntimeError(f"full success count {len(full_frame)} is less than sample size {sample_size}")

sample_frame = full_frame.sample(n=sample_size, random_state=random_state).sort_values("image_id").reset_index(drop=True)
if sample_frame["image_uri"].isna().any():
    raise RuntimeError("sample contains empty image_uri values")
if not set(sample_frame["image_id"]).issubset(set(full_frame["image_id"])):
    raise RuntimeError("sample contains image_id values not present in full dataset")

sample_dir.mkdir(parents=True, exist_ok=True)
sample_dataset_path = sample_dir / "raw.parquet"
Dataset.write(sample_frame, str(sample_dataset_path))

sample_report = {
    "source_raw_dataset_path": str(Path(result.raw_dataset_path).resolve()),
    "source_row_count": int(len(full_frame)),
    "sample_size": sample_size,
    "random_state": random_state,
}
(sample_dir / "import_report.json").write_text(
    json.dumps(sample_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("sample_raw_dataset_path:", sample_dataset_path)
print("sample_report:", sample_report)

sample_raw_dataset_path: /home/wuchaoli/codespace/ImageGallery/notebooks/.importers_test_library/default_minio_dataset/sample_1000/raw.parquet
sample_report: {'source_raw_dataset_path': '/home/wuchaoli/codespace/ImageGallery/notebooks/.importers_test_library/default_minio_dataset/full/raw.parquet', 'source_row_count': 16160, 'sample_size': 1000, 'random_state': 20260706}


In [6]:
def object_path_from_s3_uri(image_uri: str, expected_bucket: str) -> str:
    """校验 sample image_uri 属于目标 bucket，并提取 MinIO object_path。"""
    parsed = urlparse(image_uri)
    if parsed.scheme != "s3":
        raise RuntimeError(f"image_uri is not s3: {image_uri}")
    if parsed.netloc != expected_bucket:
        raise RuntimeError(f"image_uri bucket {parsed.netloc} does not match expected bucket {expected_bucket}")
    object_path = parsed.path.lstrip("/")
    if not object_path:
        raise RuntimeError(f"image_uri has empty object path: {image_uri}")
    return object_path


for label, frame in [("full", full_frame), ("sample_1000", sample_frame)]:
    prefix = f"s3://{bucket}/images/raw/"
    if not frame["image_uri"].astype(str).str.startswith(prefix).all():
        raise RuntimeError(f"{label} dataset contains image_uri outside expected prefix {prefix}")

read_check_rows = sample_frame.head(10)
for image_uri in read_check_rows["image_uri"].astype(str):
    data = storage.read_bytes(object_path_from_s3_uri(image_uri, bucket))
    if not data:
        raise RuntimeError(f"empty object bytes: {image_uri}")

print("full_row_count:", len(full_frame))
print("sample_row_count:", len(sample_frame))
print("read_check_count:", len(read_check_rows))
print("validation_status: ok")

full_row_count: 16160
sample_row_count: 1000
read_check_count: 10
validation_status: ok
